# The goal of this notebook is to use the ADNI database to determain the threshold for each biomarker:
1. p-tau 217
2. GFAP
3. NFL

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
diag_df = pd.read_csv('data/All_Subjects_DXSUM_21Jan2025.csv')
apoe_df = pd.read_csv('data/All_Subjects_APOERES_21Jan2025.csv')
pet_df = pd.read_csv('data/All_Subjects_PET_Images_21Jan2025.csv')
moca_df = pd.read_csv('data/All_Subjects_MOCA_21Jan2025.csv')
biomarker_df = pd.read_csv('data/All_Subjects_FNIHBC_BLOOD_BIOMARKER_TRAJECTORIES_21Jan2025.csv')

### Diagnosis

In [ ]:
diag_df.DIAGNOSIS.unique()


Diagnosis:
1 = CN (Cognitive Normal), 2 = MCI, 3 = Dementia

In [ ]:
for col in diag_df.columns:
    print(f'{col}: ',diag_df.iloc[50][col])

In [ ]:
diag_df.DXAD.value_counts() # 1==AD

In [ ]:
diag_df.shape

### ApoE

In [ ]:
apoe_df.GENOTYPE.value_counts()

### Biomarkers

In [ ]:
biomarker_df.head()

In [ ]:
biomarker_df['PLASMA_BIOMARKER'].value_counts()

In [ ]:
p_tau217 = biomarker_df[biomarker_df['PLASMA_BIOMARKER']=='AlzPath_plasma_ptau217']
gfap = biomarker_df[biomarker_df['PLASMA_BIOMARKER']=='QX_plasma_NfL']
nfl = biomarker_df[biomarker_df['PLASMA_BIOMARKER']=='QX_plasma_GFAP']

In [ ]:
nfl[(nfl.PTID == '011_S_0021')].EXAMDATE

I have data from different years (I think it's at least two years apart)

In [ ]:
biomraker_cols = ['PTID', 'TESTVALUE','PLASMA_BIOMARKER', 'EXAMDATE']
biomarker_df = biomarker_df[biomraker_cols]
biomarker_df = biomarker_df[biomarker_df['PLASMA_BIOMARKER'].isin(BIOMARKERS_LIST)]

In [ ]:
# Sort the dataframe by PTID, PLASMA_BIOMARKER, and EXAMDATE
df_sorted = biomarker_df.sort_values(['PTID', 'PLASMA_BIOMARKER', 'EXAMDATE'])

# Create a counter for each PTID and PLASMA_BIOMARKER combination
# This will automatically be in order of EXAMDATE due to the sorting
df_sorted['counter'] = df_sorted.groupby(['PTID', 'PLASMA_BIOMARKER']).cumcount() + 1

# Pivot the table
df = df_sorted.pivot_table(
    index='PTID',
    columns=['PLASMA_BIOMARKER', 'counter'],
    values='TESTVALUE',
    aggfunc='first'
)

# Flatten the column names
df.columns = [f'{biomarker}_{counter}' for biomarker, counter in df.columns]

# Reset the index to make PTID a column
df = df.reset_index()


In [ ]:
numeric_columns = [col for col in df.columns if col != 'PTID']
df[numeric_columns] = df[numeric_columns].apply(pd.to_numeric, errors='coerce')
# df.to_csv('data/relevent_biomarkers_df.csv')

In [ ]:
df.drop(columns='PTID').plot(kind='line', subplots=True, layout=(4,3), figsize=(15,10), title='Biomarkers over time')

In [ ]:
p_tau217 = df[['AlzPath_plasma_ptau217_1', 'AlzPath_plasma_ptau217_2', 'AlzPath_plasma_ptau217_3']]
gfap = df[['QX_plasma_GFAP_1', 'QX_plasma_GFAP_2', 'QX_plasma_GFAP_3']]
nfl = df[['QX_plasma_NfL_1', 'QX_plasma_NfL_2', 'QX_plasma_NfL_3']]

In [ ]:
corr = df.drop(columns='PTID').corr()
# Plot correlation heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", vmin=0, vmax=1)
plt.title('Correlation Heatmap of Biomarkers')
plt.savefig('figures/ADNI_biomarkers_correlation_heatmap.pdf', format='pdf', bbox_inches='tight')
plt.show()

In [ ]:
# df['AlzPath_plasma_ptau217_1'].plot(kind='hist', bins=100, title='Histogram of AlzPath_plasma_ptau217_1')]
sns.histplot(df['AlzPath_plasma_ptau217_1'], bins=100, kde=True, label='time 0', alpha=1)
sns.histplot(df['AlzPath_plasma_ptau217_2'], bins=100, kde=True, label='time 1', alpha=0.5)
sns.histplot(df['AlzPath_plasma_ptau217_3'], bins=100, kde=True, label='time 2', alpha=0.5)

plt.legend()
plt.title('p-tau disterbution over time')



In [ ]:
sns.kdeplot(df['AlzPath_plasma_ptau217_1'], label='time 0', alpha=0.5, fill=True, clip=(df['AlzPath_plasma_ptau217_1'].min(), df['AlzPath_plasma_ptau217_1'].max()))
sns.kdeplot(df['AlzPath_plasma_ptau217_2'], label='time 1', alpha=0.5, fill=True, clip=(df['AlzPath_plasma_ptau217_2'].min(), df['AlzPath_plasma_ptau217_2'].max()))
sns.kdeplot(df['AlzPath_plasma_ptau217_3'], label='time 2', alpha=0.5, fill=True, clip=(df['AlzPath_plasma_ptau217_3'].min(), df['AlzPath_plasma_ptau217_3'].max()))


plt.legend()
plt.title('p-tau disterbution over time')

plt.savefig('figures/ADNI_p-tau_disterbution_over_time.pdf', format='pdf', bbox_inches='tight')

In [ ]:
valid_ids = df[df['AlzPath_plasma_ptau217_1'] < 1]['PTID']
valid_df = df[df['PTID'].isin(valid_ids)]
valid_df.shape

In [ ]:
corr = valid_df.drop(columns='PTID').corr()
# Plot correlation heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", vmin=0, vmax=1)
plt.title('Correlation Heatmap of Biomarkers')
plt.savefig('figures/ADNI_biomarkers_correlation_heatmap_ptau_below_2.pdf', format='pdf', bbox_inches='tight')
plt.show()

In [ ]:
df['AlzPath_plasma_ptau217_1'].mean()

### Pet Scans

In [ ]:
pet_df.columns

In [ ]:
pet_df.pet_radiopharm.unique()

In [ ]:
tau_tracers = ['18F-AV1451', '18F-MK6240', '18F-PI2620']
tau_pet_scans = pet_df[pet_df['pet_radiopharm'].isin(tau_tracers)]

In [ ]:
tau_pet_scans.columns

In [ ]:
tau_pet_scans.pet_processing.unique()